# Notebook 01 — Conversion CSV → Parquet

**Orange Money — Système ML | Koceila SALEM**

Convertit le CSV brut (13 Go, 32 jours) en Parquet propre via `src/config.py`.
À lancer une seule fois. Ensuite tous les modèles lisent le Parquet.

## 0. Accès aux modules src/

In [5]:
import sys
from pathlib import Path

# Remonter à la racine du projet (ce notebook est dans notebooks/)
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print(f'Racine projet : {ROOT}')

from src import config as cfg
print(f'CSV source    : {cfg.CSV_RAW}')
print(f'Existe        : {cfg.CSV_RAW.exists()}')
print(f'Parquet cible : {cfg.PARQUET_PATH}')

Racine projet : c:\Users\RQKB6834\OneDrive - orange.com\Bureau\Alternance\ML
CSV source    : C:\Users\RQKB6834\OneDrive - orange.com\Bureau\Alternance\ML\data\raw\OM_Koceila.csv
Existe        : True
Parquet cible : C:\Users\RQKB6834\OneDrive - orange.com\Bureau\Alternance\ML\data\processed\OM_clean.parquet


In [6]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import time, os, warnings
warnings.filterwarnings('ignore')

if cfg.CSV_RAW.exists():
    print(f'Taille CSV : {os.path.getsize(cfg.CSV_RAW)/(1024**3):.2f} Go')

Taille CSV : 13.21 Go


## 1. Validation des colonnes

In [7]:
df_peek = pd.read_csv(cfg.CSV_RAW, sep=cfg.CSV_SEP, encoding=cfg.CSV_ENCODING,
                      nrows=1000, low_memory=False)

presentes  = [c for c in cfg.COLS_PARQUET if c in df_peek.columns]
manquantes = [c for c in cfg.COLS_PARQUET if c not in df_peek.columns]

print(f'Présentes  : {len(presentes)}/{len(cfg.COLS_PARQUET)}')
if manquantes:
    print(f'Manquantes : {manquantes}')
    print('   -> Retire-les de cfg.COLS_PARQUET dans src/config.py')
else:
    print('Toutes les colonnes utiles sont présentes')

Présentes  : 45/45
Toutes les colonnes utiles sont présentes


## 2. Conversion par chunks

In [9]:
def nettoyer_chunk(chunk):
    """Typage stable d'un chunk : force les types pour un schéma constant."""
    # Numériques -> float64 systématique
    for col in cfg.COLS_NUM:
        if col in chunk.columns:
            chunk[col] = pd.to_numeric(chunk[col], errors='coerce').astype('float64')

    # Dates -> datetime64
    for col in cfg.COLS_DATE:
        if col in chunk.columns:
            chunk[col] = pd.to_datetime(chunk[col], errors='coerce', dayfirst=True)

    # TOUTES les autres colonnes -> string (object) pour éviter
    # qu'une colonne vide soit float dans un chunk et texte dans un autre
    cols_string = [c for c in chunk.columns
                   if c not in cfg.COLS_NUM and c not in cfg.COLS_DATE]
    for col in cols_string:
        chunk[col] = chunk[col].astype(str).replace('nan', None)

    return chunk


cfg.PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)

print('Conversion en cours (plusieurs minutes)...')
debut = time.time()
writer = None
schema_ref = None   # schéma figé du premier chunk
n_total = 0

reader = pd.read_csv(
    cfg.CSV_RAW, sep=cfg.CSV_SEP, encoding=cfg.CSV_ENCODING,
    chunksize=cfg.CHUNK_SIZE, low_memory=False,
    usecols=lambda c: c in presentes
)

for i, chunk in enumerate(reader, 1):
    chunk = nettoyer_chunk(chunk)
    n_total += len(chunk)

    if schema_ref is None:
        # Premier chunk : on fige le schéma et on réordonne les colonnes
        table = pa.Table.from_pandas(chunk, preserve_index=False)
        schema_ref = table.schema
        writer = pq.ParquetWriter(cfg.PARQUET_PATH, schema_ref, compression='snappy')
    else:
        # Chunks suivants : on force le même schéma
        table = pa.Table.from_pandas(chunk, preserve_index=False, schema=schema_ref)

    writer.write_table(table)
    if i % 5 == 0:
        print(f'  Chunk {i:3d} | {n_total:,} lignes | {time.time()-debut:.0f}s')

if writer:
    writer.close()

duree = time.time() - debut
taille_pq  = cfg.PARQUET_PATH.stat().st_size / (1024**3)
taille_csv = os.path.getsize(cfg.CSV_RAW) / (1024**3)
print(f'\nTERMINÉ en {duree:.0f}s')
print(f'   Lignes      : {n_total:,}')
print(f'   Parquet     : {taille_pq:.2f} Go')
print(f'   Compression : {taille_csv/taille_pq:.1f}x')

Conversion en cours (plusieurs minutes)...
  Chunk   5 | 2,500,000 lignes | 105s
  Chunk  10 | 5,000,000 lignes | 214s
  Chunk  15 | 7,500,000 lignes | 316s
  Chunk  20 | 10,000,000 lignes | 409s
  Chunk  25 | 12,500,000 lignes | 499s
  Chunk  30 | 15,000,000 lignes | 589s
  Chunk  35 | 17,500,000 lignes | 684s
  Chunk  40 | 20,000,000 lignes | 792s
  Chunk  45 | 22,500,000 lignes | 888s
  Chunk  50 | 25,000,000 lignes | 983s

TERMINÉ en 1004s
   Lignes      : 25,456,467
   Parquet     : 2.21 Go
   Compression : 6.0x


## 3. Vérification

In [10]:
from src.data_loader import load_parquet

df_check = load_parquet(columns=[cfg.COL_DATE, cfg.COL_STATUT, cfg.COL_SERVICE])

print('\nDistribution TRANSFER_STATUS :')
print(df_check[cfg.COL_STATUT].value_counts())
print('\nDistribution SERVICE_TYPE :')
print(df_check[cfg.COL_SERVICE].value_counts())
print('\nParquet prêt -> notebooks/M1_fraude/02_M1_isolation_forest.ipynb')

Chargé en 3.3s
Dimensions : 25,456,467 lignes × 3 colonnes
RAM        : 2.68 Go
Plage      : 2025-08-29 10:35:20 → 2025-09-30 23:59:59
Jours      : 32

Distribution TRANSFER_STATUS :
TRANSFER_STATUS
TS     23967725
TF      1478546
TPI       10017
TI          167
A1           12
Name: count, dtype: int64

Distribution SERVICE_TYPE :
SERVICE_TYPE
RC            7973510
MERCHPAY      6028266
CASHOUT       4325031
CASHIN        4087453
P2P           2922121
ROLLBACK        71495
TXNCORRECT      30753
ENT2REG         11630
B2BCASHOUT       4546
O2C               642
OPTW              626
DELSCR            201
COUTBYCODE        183
ENT2UNPASS          7
STOCK               2
ENT2UNREG           1
Name: count, dtype: int64

Parquet prêt -> notebooks/M1_fraude/02_M1_isolation_forest.ipynb
